# 3rd Place - Team G & B & D & T Final Solution - LB 0.604!
This notebook is the 3rd place final submission for Team G&B&D&T for Kaggle's OTTO RecSys competition. Team members are Giba ( @titericz ), Benny ( @benediktschifferer ), Chris Deotte ( @cdeotte ), and Theo ( @theoviel ). This solution is an ensemble of 3 single XGB reranker models. Discussion explaining this solution is [here][1], [here][2], and [here][3].

[1]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/386497
[2]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/383013
[3]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/382975

In [ ]:
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter

pd.options.display.max_colwidth = 500

In [ ]:
FILES = [
    "/kaggle/input/otto-comp-single-models/submission_chris_v186v406v412.csv", # Chris 0.601
    "/kaggle/input/otto-comp-single-models/submission_benny_601.csv",      # Benny 0.601
    "/kaggle/input/otto-comp-single-models/submission_theo_6029.csv"       # Theo 0.603
]

WEIGHTS = [1, 1, 3]

In [ ]:
subs = [
    pd.read_csv(f).sort_values(['session_type']).reset_index(drop=True) for f in FILES
]

for s in subs:
    assert len(s) == len(subs[0])

In [ ]:
for s, f in zip(subs, FILES):
    print(f.split('/')[-1])
    assert len(s) == len(subs[0])
    display(s.head(3))
    print()

In [ ]:
for idx in range(9):
    for j in range(len(subs)):
        for k in range(j):
            p1 = np.array(sorted(subs[j]['labels'][idx].split(' '))).astype(int)
            p2 = np.array(sorted(subs[k]['labels'][idx].split(' '))).astype(int)
            sim = len((set(p1).intersection(set(p2)))) / 20
            print(f'Similarity of row {subs[0]["session_type"][idx]} between subs {j} & {k} : {sim :.3f}')
    print()

In [ ]:
blend = []
for idx in tqdm(range(len(subs[0]))):    
    amap = Counter()
    for sub, sub_w in zip(subs, WEIGHTS):
        for w, i in enumerate(sub["labels"][idx].split(' ')):
            amap[i] += (sub_w * (20 - w))

    aid = ' '.join([aid_ for aid_, _ in amap.most_common(20)])
    blend.append(aid)

In [ ]:
sub = subs[0].copy()
sub['labels'] = blend
sub.to_csv('submission.csv', index=False)

sub.head(12)

In [ ]:
for idx in range(3):
    for j in range(len(subs)):
        p1 = np.array(sorted(subs[j]['labels'][idx].split(' '))).astype(int)
        p2 = np.array(sorted(sub['labels'][idx].split(' '))).astype(int)
        sim = len((set(p1).intersection(set(p2)))) / 20
        print(f'Similarity of row {subs[0]["session_type"][idx]} between sub {j} & blend : {sim :.3f}')
    print()

Done ! 